# Arquitectura OLAP en Spotify (Dataset Real 2024) 🎵
## Contexto:
Eres el Arquitecto de Datos Principal en una disquera. El equipo de ingeniería extrajo un dump transaccional masivo con las canciones más reproducidas de Spotify en 2024. El archivo plano es ancho, redundante y poco óptimo para análisis.

Tu misión es aplicar una tubería ETL para convertir este archivo crudo en un Esquema de Estrella en memoria, y posteriormente responder a 10 preguntas críticas del equipo de analítica utilizando el Cubo OLAP.

In [29]:
import pandas as pd
import numpy as np

# EXTRACT: Leemos directamente el archivo crudo desde el repositorio
url_dataset = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-III/datasets/Most%20Streamed%20Spotify%20Songs%202024.csv"

# Para evitar errores de codificación con caracteres especiales en nombres de artistas
spotify_raw = pd.read_csv(url_dataset, encoding='latin-1')

print("Muestra de datos crudos (OLTP):")
display(spotify_raw.head(3))
print("Columnas disponibles:", spotify_raw.columns.tolist())


Muestra de datos crudos (OLTP):


,Track,Album Name,Artist,Release Date,ISRC,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,...,SiriusXM Spins,Deezer Playlist Count,Deezer Playlist Reach,Amazon Playlist Count,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Explicit Track
0,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,4/26/2024,QM24S2402528,1,725.4,"390,470,936","30,716","196,631,588",...,684,62.0,"17,598,718",114.0,"18,004,655","22,931","4,818,457","2,669,262",NaN,0
1,Not Like Us,Not Like Us,Kendrick Lamar,5/4/2024,USUG12400910,2,545.9,"323,703,884","28,113","174,597,137",...,3,67.0,"10,422,430",111.0,"7,780,028","28,444","6,623,075","1,118,279",NaN,1
2,i like the way you kiss me,I like the way you kiss me,Artemas,3/19/2024,QZJ842400387,3,538.4,"601,309,283","54,331","211,607,669",...,536,136.0,"36,321,847",172.0,"5,022,621","5,639","7,208,651","5,285,340",NaN,0


Columnas disponibles: ['Track', 'Album Name', 'Artist', 'Release Date', 'ISRC', 'All Time Rank', 'Track Score', 'Spotify Streams', 'Spotify Playlist Count', 'Spotify Playlist Reach', 'Spotify Popularity', 'YouTube Views', 'YouTube Likes', 'TikTok Posts', 'TikTok Likes', 'TikTok Views', 'YouTube Playlist Reach', 'Apple Music Playlist Count', 'AirPlay Spins', 'SiriusXM Spins', 'Deezer Playlist Count', 'Deezer Playlist Reach', 'Amazon Playlist Count', 'Pandora Streams', 'Pandora Track Stations', 'Soundcloud Streams', 'Shazam Counts', 'TIDAL Popularity', 'Explicit Track']


### Fase 2: Transform & Load (Construyendo el Almacén)

Observa las columnas del dataset. Tienes texto (Track, Artist, Date) mezclado con métricas puras (Spotify Streams, YouTube Views). 

**Reto Arquitectónico:**
1. Crea una **Dimensión Tiempo (`dim_tiempo`)**: Extrae las columnas relacionadas al año, mes y día de lanzamiento. Elimina duplicados y crea un `id_tiempo`.
2. Crea una **Dimensión Pista (`dim_pista`)**: Extrae el nombre del track, el artista y si es explícita. Crea un `id_pista`.
3. Crea la **Tabla de Hechos (`fact_streams`)**: Cruza los IDs y conserva **únicamente** métricas numéricas (ej. Spotify Streams, YouTube Views, Track Score) junto con las llaves foráneas. NO debe haber texto aquí.

In [30]:
# --- ESCRIBE TU PIPELINE ETL AQUÍ ---

# 1. Crear dim_tiempo
dim_tiempo = spotify_raw[['Release Date']].drop_duplicates().reset_index(drop=True)
dim_tiempo['id_tiempo'] = dim_tiempo.index + 1

fechas_dt = pd.to_datetime(dim_tiempo['Release Date'], errors='coerce')
dim_tiempo['Year'] = fechas_dt.dt.year
dim_tiempo['Month'] = fechas_dt.dt.month
dim_tiempo['Day'] = fechas_dt.dt.day

print(dim_tiempo.head(10))

  Release Date  id_tiempo  Year  Month  Day
0    4/26/2024          1  2024      4   26
1     5/4/2024          2  2024      5    4
2    3/19/2024          3  2024      3   19
3    1/12/2023          4  2023      1   12
4    5/31/2024          5  2024      5   31
5   11/10/2023          6  2023     11   10
6    1/18/2024          7  2024      1   18
7     2/2/2024          8  2024      2    2
8     6/9/2024          9  2024      6    9
9    5/23/2024         10  2024      5   23


In [31]:
# 2. Crear dim_pista
dim_pista = spotify_raw[['Track', 'Artist', 'Explicit Track']].drop_duplicates().reset_index(drop=True)
dim_pista['id_pista'] = dim_pista.index + 1

print(dim_pista.head())

                        Track          Artist  Explicit Track  id_pista
0         MILLION DOLLAR BABY   Tommy Richman               0         1
1                 Not Like Us  Kendrick Lamar               1         2
2  i like the way you kiss me         Artemas               0         3
3                     Flowers     Miley Cyrus               0         4
4                     Houdini          Eminem               1         5


In [32]:
# 3. Crear fact_streams

spotify_raw.columns = spotify_raw.columns.str.strip()

df_temp = spotify_raw.merge(dim_tiempo[['Release Date', 'id_tiempo']], on='Release Date', how='left')
df_temp = df_temp.merge(dim_pista[['Track', 'Artist', 'Explicit Track', 'id_pista']], on=['Track', 'Artist', 'Explicit Track'], how='left')

fact_streams = df_temp[['id_tiempo', 'id_pista', 'Spotify Streams', 'YouTube Views', 'Track Score']].copy()

for col in ['Spotify Streams', 'YouTube Views']:
    fact_streams[col] = fact_streams[col].astype(str).str.replace(',', '', regex=False)
    fact_streams[col] = pd.to_numeric(fact_streams[col], errors='coerce')

fact_streams['Track Score'] = pd.to_numeric(fact_streams['Track Score'], errors='coerce')

print(fact_streams.head())

   id_tiempo  id_pista  Spotify Streams  YouTube Views  Track Score
0          1         1     3.904709e+08   8.427475e+07        725.4
1          2         2     3.237039e+08   1.163470e+08        545.9
2          3         3     6.013093e+08   1.225991e+08        538.4
3          4         4     2.031281e+09   1.096101e+09        444.9
4          5         5     1.070349e+08   7.737396e+07        423.3


In [33]:
# 4. (LOAD) Reconstruir el Cubo en memoria para las consultas
cubo_olap = fact_streams.merge(dim_tiempo, on='id_tiempo', how='left') \
             .merge(dim_pista, on='id_pista', how='left')

print("Cubo OLAP (OLTP):")
display(cubo_olap.head(3))

Cubo OLAP (OLTP):


,id_tiempo,id_pista,Spotify Streams,YouTube Views,Track Score,Release Date,Year,Month,Day,Track,Artist,Explicit Track
0,1,1,390470936.0,84274754.0,725.4,4/26/2024,2024,4,26,MILLION DOLLAR BABY,Tommy Richman,0
1,2,2,323703884.0,116347040.0,545.9,5/4/2024,2024,5,4,Not Like Us,Kendrick Lamar,1
2,3,3,601309283.0,122599116.0,538.4,3/19/2024,2024,3,19,i like the way you kiss me,Artemas,0


### Explica porqué solo hay 2 tablas dimensiones o si podrías incluir alguna otra. \

Actualmente solo existen 2 dimensiones (`dim_tiempo` y `dim_pista`) porque representan los dos factores que se pidieron el tiempo cuando fue lanzada la cancion y toda la informacion musical de la pista.

Se podrian incluir dimensiones como el artista, o la plataforma por la cual fue lanzada o reproducida la cancion.

### Fase 3: Consultas Analíticas (10 Retos)

El equipo de analítica te ha enviado 10 peticiones. Para cada una, debes:
1. Escribir el código en Pandas usando la matriz `cubo_olap` que construiste.
2. Identificar y escribir en un comentario qué tipo de operación OLAP estás aplicando: **Roll-up, Drill-down, Slice o Dice**.

In [39]:
# RETO 1: El CEO quiere ver el total acumulado de 'Spotify Streams' agrupado por 'Artist'.
# Operación OLAP:  Es una operacion Roll-UP
respuesta_1 = cubo_olap.groupby('Artist')['Spotify Streams'].sum().reset_index()

# RETO 2: El equipo de marketing quiere ver todas las métricas únicamente de las canciones lanzadas en el año 2024.
# Operación OLAP: Operacion Slice por la dimension de año
respuesta_2 = cubo_olap[cubo_olap['Year'] == 2024]
#
# # RETO 3: Finanzas requiere el total de 'Spotify Streams' desglosado primero por Año de Lanzamiento (Year) y luego detallado por Mes de Lanzamiento (Month).
# # Operación OLAP: Operacion Drill Down por los niveles de jerarquia en los años
respuesta_3 = cubo_olap.groupby(['Year', 'Month'])['Spotify Streams'].sum().reset_index()
#
# # RETO 4: Auditoría te pide extraer el rendimiento de las canciones que son explícitas Y que además fueron lanzadas en el año 2024.
# # Operación OLAP: Dice, ya que filtra por medio de dos dimensiones
respuesta_4 = cubo_olap[(cubo_olap['Explicit Track'] == 1) & (cubo_olap['Year'] == 2024)]
#
# # RETO 5: Queremos disminuir el nivel de detalle y ver solamente el promedio del 'Track Score' agrupado por Año de Lanzamiento.
# # Operación OLAP: roll Up por reducirlo al promedio del Track Score
respuesta_5 = cubo_olap.groupby('Year')['Track Score'].mean().reset_index()

print("Resultado de la consulta OLAP:")
print(respuesta_1.head(3), "\n")
print(respuesta_2.head(3), "\n")
print(respuesta_3.head(3), "\n")
print(respuesta_4.head(3), "\n")
print(respuesta_5.head(3), "\n")


Resultado de la consulta OLAP:
        Artist  Spotify Streams
0         "XY"     0.000000e+00
1    $OHO BANI     5.406556e+07
2  $uicideboy$     1.697447e+09 

   id_tiempo  id_pista  Spotify Streams  YouTube Views  Track Score  \
0          1         1      390470936.0     84274754.0        725.4   
1          2         2      323703884.0    116347040.0        545.9   
2          3         3      601309283.0    122599116.0        538.4   

  Release Date  Year  Month  Day                       Track          Artist  \
0    4/26/2024  2024      4   26         MILLION DOLLAR BABY   Tommy Richman   
1     5/4/2024  2024      5    4                 Not Like Us  Kendrick Lamar   
2    3/19/2024  2024      3   19  i like the way you kiss me         Artemas   

   Explicit Track  
0               0  
1               1  
2               0   

   Year  Month  Spotify Streams
0  1987      7     1.879386e+09
1  1991      9     2.021910e+09
2  1994      1     1.810650e+09 

   id_tiempo  id_pist

In [40]:
# RETO 6: Muestra el comportamiento analítico aislando únicamente los datos del artista "Bad Bunny".
# Operación OLAP: Slice corte en una sola dimension
respuesta_6 = cubo_olap[cubo_olap['Artist'] == 'Bad Bunny']

# RETO 7: Aumenta el nivel de detalle: Muestra el total de 'YouTube Views' agrupado por Artista, y dentro de cada Artista, desglosado por Nombre de la Canción (Track).
# Operación OLAP: Drill Down
respuesta_7 = cubo_olap.groupby(['Artist', 'Track'])['YouTube Views'].sum().reset_index()

# RETO 8: Extrae un sub-cubo que contenga únicamente canciones lanzadas en el Mes 12 (Diciembre) del Año 2023.
# Operación OLAP: Dice
respuesta_8 = cubo_olap[(cubo_olap['Year'] == 2023) & (cubo_olap['Month'] == 12)]

# RETO 9: Resume la información calculando el total histórico de 'Spotify Streams' de todos los tiempos, agrupado por Mes (para saber en qué mes históricamente se lanza la música más exitosa).
# Operación OLAP: Roll Up
respuesta_9 = cubo_olap.groupby('Month')['Spotify Streams'].sum().reset_index()

# RETO 10: Extrae un bloque de información filtrando las canciones de "Taylor Swift" que NO sean explícitas y hayan salido en 2024.
# Operación OLAP: Dice
respuesta_10 = cubo_olap[
    (cubo_olap['Artist'] == 'Taylor Swift') &
    (cubo_olap['Year'] == 2024) &
    (cubo_olap['Explicit Track'] == 0)
]

print("Resultado de la consulta OLAP:")
print(respuesta_6.head(3), "\n")
print(respuesta_7.head(3), "\n")
print(respuesta_8.head(3), "\n")
print(respuesta_9.head(3), "\n")
print(respuesta_10.head(3), "\n")

Resultado de la consulta OLAP:
     id_tiempo  id_pista  Spotify Streams  YouTube Views  Track Score  \
63          57        61     1.811991e+09   1.516863e+09        184.8   
88          57        86     1.514240e+09   1.366181e+09        163.1   
102         91       100     4.781509e+08   1.133797e+08        150.7   

    Release Date  Year  Month  Day               Track     Artist  \
63      5/6/2022  2022      5    6     Me Porto Bonito  Bad Bunny   
88      5/6/2022  2022      5    6  Titï¿½ï¿½ Me Pregu  Bad Bunny   
102   10/13/2023  2023     10   13              MONACO  Bad Bunny   

     Explicit Track  
63                1  
88                1  
102               1   

        Artist                                              Track  \
0         "XY"                        Come and Say - Original Mix   
1    $OHO BANI                          ZEIT, DASS SICH WAS DREHT   
2  $uicideboy$  ...And to Those I Love, Thanks for Sticking Ar...   

   YouTube Views  
0    41515914